In [ ]:
# In[1]:

import pandas as pd
import pytz
from datetime import datetime

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# helper to convert unix seconds to tz-aware ISO string
def ts_to_iso(ts):
    if pd.isna(ts):
        return None
    try:
        return datetime.fromtimestamp(int(ts), tz).isoformat()
    except Exception:
        return None

def summarize_dataframe(df, name_field):
    # total rows
    total_rows = int(df.shape[0])
    # min/max timestamps (as ints and tz-aware ISO)
    if total_rows == 0 or 'timestamp' not in df.columns:
        min_ts_int = None
        max_ts_int = None
        min_ts_iso = None
        max_ts_iso = None
    else:
        min_ts_int = int(df['timestamp'].min())
        max_ts_int = int(df['timestamp'].max())
        min_ts_iso = ts_to_iso(min_ts_int)
        max_ts_iso = ts_to_iso(max_ts_int)
    # top 20 cmdb_id counts
    if 'cmdb_id' in df.columns and total_rows > 0:
        top_cmdb = df['cmdb_id'].value_counts().head(20).reset_index()
        top_cmdb.columns = ['cmdb_id', 'count']
        top_cmdb_list = top_cmdb.to_dict(orient='records')
    else:
        top_cmdb_list = []
    # top 20 names counts (kpi_name / trace_name / log_name / message)
    if name_field in df.columns and total_rows > 0:
        top_names = df[name_field].value_counts().head(20).reset_index()
        top_names.columns = [name_field, 'count']
        top_names_list = top_names.to_dict(orient='records')
    else:
        top_names_list = []
    # example rows (first 5) as dicts
    examples = df.head(5).to_dict(orient='records')
    # assemble compact summary
    summary = {
        'rows': total_rows,
        'min_timestamp': {'unix': min_ts_int, 'tz_iso': min_ts_iso},
        'max_timestamp': {'unix': max_ts_int, 'tz_iso': max_ts_iso},
        'top_cmdb_id_counts': top_cmdb_list,
        'top_name_counts': top_names_list,
        'example_rows': examples
    }
    return summary

# Load files
df = pd.read_csv('metric.csv')          # metric
df_t = pd.read_csv('trace.csv')         # trace
df_l = pd.read_csv('log.csv')           # log
df_e = pd.read_csv('error_logs.csv')    # error logs

# Ensure timestamp numeric where present
for d in (df, df_t, df_l, df_e):
    if 'timestamp' in d.columns:
        d['timestamp'] = pd.to_numeric(d['timestamp'], errors='coerce').astype('Int64')

# Summaries
metric_summary = summarize_dataframe(df, 'kpi_name')
trace_summary = summarize_dataframe(df_t, 'trace_name')
log_summary = summarize_dataframe(df_l, 'log_name')
# For error_logs, top distinct 'message' values
error_logs_summary = summarize_dataframe(df_e, 'message')

# Final JSON-serializable result
result = {
    'metric_summary': metric_summary,
    'trace_summary': trace_summary,
    'log_summary': log_summary,
    'error_logs_summary': error_logs_summary
}

result

```
Out[1]:
```
```python
# Use the previously loaded 'result' summary produced earlier to build a plain-English summary string.
# This cell outputs a concise human-readable summary.

metric = result['metric_summary']
trace = result['trace_summary']
log = result['log_summary']
err = result['error_logs_summary']

def top_list_names(entries, n=3):
    return ', '.join([e[next(iter(e))] for e in entries[:n]])

summary = (
    f"metric.csv: {metric['rows']} rows; timestamps {metric['min_timestamp']['tz_iso']} to {metric['max_timestamp']['tz_iso']}. "
    f"Top cmdb_ids (by rows): {[c['cmdb_id'] for c in metric['top_cmdb_id_counts'][:5]]}. "
    f"Top KPI names (examples): {top_list_names(metric['top_name_counts'], 3)}. "
    f"Example rows show metrics like {metric['example_rows'][:3]}.\n\n"
    f"trace.csv: {trace['rows']} rows; timestamps {trace['min_timestamp']['tz_iso']} to {trace['max_timestamp']['tz_iso']}. "
    f"Top cmdb_ids: {[c['cmdb_id'] for c in trace['top_cmdb_id_counts'][:5]]}. "
    f"Top trace names (examples): {top_list_names(trace['top_name_counts'], 3)}. "
    f"Example rows show traces like {trace['example_rows'][:3]}.\n\n"
    f"log.csv: {log['rows']} rows; timestamps {log['min_timestamp']['tz_iso']} to {log['max_timestamp']['tz_iso']}. "
    f"Top cmdb_ids: {[c['cmdb_id'] for c in log['top_cmdb_id_counts'][:5]]}. "
    f"Top log names: {[n['log_name'] for n in log['top_name_counts'][:5]]}. "
    f"Example rows: {log['example_rows'][:3]}.\n\n"
    f"error_logs.csv: {err['rows']} rows; timestamps {err['min_timestamp']['tz_iso']} to {err['max_timestamp']['tz_iso']}. "
    f"Top cmdb_ids: {[c['cmdb_id'] for c in err['top_cmdb_id_counts'][:5]]}. "
    f"Top messages: {[m['message'] for m in err['top_name_counts'][:2]]}. "
    f"Example rows: {err['example_rows'][:3]}."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric_summary': {'rows': 146292, 'min_timestamp': {'unix': 1647806400, 'tz_iso': '2022-03-21T04:00:00+08:00'}, 'max_timestamp': {'unix': 1647808140, 'tz_iso': '2022-03-21T04:29:00+08:00'}, 'top_cmdb_id_counts': [{'cmdb_id': 'adservice', 'count': 10230}, {'cmdb_id': 'adservice2', 'count': 9990}, {'cmdb_id': 'frontend2-0', 'count': 5040}, {'cmdb_id': 'frontend-0', 'count': 4920}, {'cmdb_id': 'frontend-1', 'count': 4680}, {'cmdb_id': 'frontend-2', 'count': 4500}, {'cmdb_id': 'checkoutservice-0', 'count': 3900}, {'cmdb_id': 'checkoutservice-2', 'count': 3660}, {'cmdb_id': 'checkoutservice-1', 'count': 3540}, {'cmdb_id': 'productcatalogservice-0', 'count': 3480}, {'cmdb_id': 'productcatalogservice-2', 'count': 3480}, {'cmdb_id': 'productcatalogservice-1', 'count': 3360}, {'cmdb_id': 'checkoutservice2-0', 'count': 3300}, {'cmdb_id': 'adservice2-0', 'count': 3060}, {'cmdb_id': 'cartservice-1', 'count': 3000}, {'cmdb_id': 'productcatalogservice2-0', 'count': 2880}, {'cmdb_id': 'cartservice-0', 'count': 2760}, {'cmdb_id': 'cartservice-2', 'count': 2640}, {'cmdb_id': 'cartservice2-0', 'count': 2640}, {'cmdb_id': 'adservice-0', 'count': 2580}], 'top_name_counts': [{'kpi_name': 'container.node-5.container_network_transmit_packets_dropped.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_transmit_packets.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_receive_MB.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_receive_errors.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_transmit_errors.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_receive_packets.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_transmit_MB.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_network_receive_packets_dropped.eth0', 'count': 1050}, {'kpi_name': 'container.node-5.container_tasks_state.sleeping', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_io_current./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_usage_MB', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_swap', 'count': 990}, {'kpi_name': 'container.node-5.container_memory_rss', 'count': 990}, {'kpi_name': 'container.node-5.container_cpu_user_seconds', 'count': 990}, {'kpi_name': 'container.node-5.container_cpu_usage_seconds', 'count': 990}, {'kpi_name': 'container.node-5.container_cpu_system_seconds', 'count': 990}, {'kpi_name': 'container.node-5.container_cpu_load_average_10s', 'count': 990}, {'kpi_name': 'container.node-5.container_spec_cpu_quota', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_reads_merged./dev/vda1', 'count': 990}, {'kpi_name': 'container.node-5.container_fs_reads./dev/vda1', 'count': 990}], 'example_rows': [{'timestamp': 1647806400, 'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.count', 'value': 80.0}, {'timestamp': 1647806400, 'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.mrt', 'value': 0.002475}, {'timestamp': 1647806400, 'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.rr', 'value': 100.0}, {'timestamp': 1647806400, 'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.sr', 'value': 78.75}, {'timestamp': 1647806400, 'cmdb_id': 'adservice', 'kpi_name': 'app.http.count', 'value': 24.0}]}, 'trace_summary': {'rows': 29944, 'min_timestamp': {'unix': 1647806400, 'tz_iso': '2022-03-21T04:00:00+08:00'}, 'max_timestamp': {'unix': 1647808140, 'tz_iso': '2022-03-21T04:29:00+08:00'}, 'top_cmdb_id_counts': [{'cmdb_id': 'frontend-2', 'count': 2572}, {'cmdb_id': 'frontend-0', 'count': 2500}, {'cmdb_id': 'frontend-1', 'count': 2472}, {'cmdb_id': 'checkoutservice-1', 'count': 1696}, {'cmdb_id': 'checkoutservice-0', 'count': 1572}, {'cmdb_id': 'checkoutservice-2', 'count': 1564}, {'cmdb_id': 'productcatalogservice-1', 'count': 1076}, {'cmdb_id': 'productcatalogservice-2', 'count': 1076}, {'cmdb_id': 'productcatalogservice-0', 'count': 1072}, {'cmdb_id': 'frontend2-0', 'count': 1068}, {'cmdb_id': 'checkoutservice2-0', 'count': 1044}, {'cmdb_id': 'recommendationservice-2', 'count': 840}, {'cmdb_id': 'recommendationservice-1', 'count': 840}, {'cmdb_id': 'recommendationservice-0', 'count': 840}, {'cmdb_id': 'cartservice-1', 'count': 728}, {'cmdb_id': 'cartservice-0', 'count': 724}, {'cmdb_id': 'cartservice-2', 'count': 704}, {'cmdb_id': 'currencyservice-2', 'count': 648}, {'cmdb_id': 'currencyservice-0', 'count': 644}, {'cmdb_id': 'currencyservice-1', 'count': 644}], 'top_name_counts': [{'trace_name': 'trace.from_frontend-2.row_count', 'count': 583}, {'trace_name': 'trace.from_frontend-2.error_rate', 'count': 583}, {'trace_name': 'trace.from_frontend-2.duration_p95', 'count': 583}, {'trace_name': 'trace.from_frontend-2.duration_mean', 'count': 583}, {'trace_name': 'trace.from_frontend-0.duration_mean', 'count': 565}, {'trace_name': 'trace.from_frontend-0.duration_p95', 'count': 565}, {'trace_name': 'trace.from_frontend-0.error_rate', 'count': 565}, {'trace_name': 'trace.from_frontend-0.row_count', 'count': 565}, {'trace_name': 'trace.from_frontend-1.row_count', 'count': 558}, {'trace_name': 'trace.from_frontend-1.error_rate', 'count': 558}, {'trace_name': 'trace.from_frontend-1.duration_p95', 'count': 558}, {'trace_name': 'trace.from_frontend-1.duration_mean', 'count': 558}, {'trace_name': 'trace.self.row_count', 'count': 463}, {'trace_name': 'trace.self.error_rate', 'count': 463}, {'trace_name': 'trace.self.duration_p95', 'count': 463}, {'trace_name': 'trace.self.duration_mean', 'count': 463}, {'trace_name': 'trace.from_checkoutservice-1.duration_mean', 'count': 334}, {'trace_name': 'trace.from_checkoutservice-1.duration_p95', 'count': 334}, {'trace_name': 'trace.from_checkoutservice-1.error_rate', 'count': 334}, {'trace_name': 'trace.from_checkoutservice-1.row_count', 'count': 334}], 'example_rows': [{'timestamp': 1647806400, 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.duration_mean', 'value': 1.4e-05}, {'timestamp': 1647806400, 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.duration_p95', 'value': 1.5e-05}, {'timestamp': 1647806400, 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.error_rate', 'value': 0.0}, {'timestamp': 1647806400, 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-0.row_count', 'value': 5.0}, {'timestamp': 1647806400, 'cmdb_id': 'adservice-0', 'trace_name': 'trace.from_frontend-1.duration_mean', 'value': 1.4e-05}]}, 'log_summary': {'rows': 1706, 'min_timestamp': {'unix': 1647806400, 'tz_iso': '2022-03-21T04:00:00+08:00'}, 'max_timestamp': {'unix': 1647808140, 'tz_iso': '2022-03-21T04:29:00+08:00'}, 'top_cmdb_id_counts': [{'cmdb_id': 'adservice-0', 'count': 60}, {'cmdb_id': 'adservice-1', 'count': 60}, {'cmdb_id': 'adservice-2', 'count': 60}, {'cmdb_id': 'cartservice-0', 'count': 60}, {'cmdb_id': 'cartservice-1', 'count': 60}, {'cmdb_id': 'cartservice-2', 'count': 60}, {'cmdb_id': 'currencyservice-0', 'count': 60}, {'cmdb_id': 'frontend-0', 'count': 60}, {'cmdb_id': 'currencyservice-1', 'count': 60}, {'cmdb_id': 'currencyservice-2', 'count': 60}, {'cmdb_id': 'frontend-1', 'count': 60}, {'cmdb_id': 'productcatalogservice-1', 'count': 60}, {'cmdb_id': 'productcatalogservice-2', 'count': 60}, {'cmdb_id': 'productcatalogservice-0', 'count': 60}, {'cmdb_id': 'frontend-2', 'count': 60}, {'cmdb_id': 'shippingservice-1', 'count': 60}, {'cmdb_id': 'shippingservice-2', 'count': 60}, {'cmdb_id': 'shippingservice-0', 'count': 60}, {'cmdb_id': 'recommendationservice-2', 'count': 60}, {'cmdb_id': 'recommendationservice-0', 'count': 60}], 'top_name_counts': [{'log_name': 'log.error_count', 'count': 853}, {'log_name': 'log.row_count', 'count': 853}], 'example_rows': [{'timestamp': 1647806400, 'cmdb_id': 'adservice-0', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1647806400, 'cmdb_id': 'adservice-0', 'log_name': 'log.row_count', 'value': 390.0}, {'timestamp': 1647806400, 'cmdb_id': 'adservice-1', 'log_name': 'log.error_count', 'value': 0.0}, {'timestamp': 1647806400, 'cmdb_id': 'adservice-1', 'log_name': 'log.row_count', 'value': 36.0}, {'timestamp': 1647806400, 'cmdb_id': 'adservice-2', 'log_name': 'log.error_count', 'value': 0.0}]}, 'error_logs_summary': {'rows': 542, 'min_timestamp': {'unix': 1647806404, 'tz_iso': '2022-03-21T04:00:04+08:00'}, 'max_timestamp': {'unix': 1647808199, 'tz_iso': '2022-03-21T04:29:59+08:00'}, 'top_cmdb_id_counts': [{'cmdb_id': 'frontend-2', 'count': 234}, {'cmdb_id': 'frontend-1', 'count': 158}, {'cmdb_id': 'frontend-0', 'count': 150}], 'top_name_counts': [{'message': 'severity: warning, message: failed to retrieve ads', 'count': 542}], 'example_rows': [{'timestamp': 1647806404, 'cmdb_id': 'frontend-0', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': 1647806408, 'cmdb_id': 'frontend-2', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': 1647806412, 'cmdb_id': 'frontend-1', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': 1647806414, 'cmdb_id': 'frontend-0', 'message': 'severity: warning, message: failed to retrieve ads'}, {'timestamp': 1647806417, 'cmdb_id': 'frontend-2', 'message': 'severity: warning, message: failed to retrieve ads'}]}}```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz
from datetime import datetime

# reuse existing 'df' if available, else load metric.csv
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')

# ensure timestamp numeric and value numeric
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# parameters
start_ts = 1647806400
end_ts = 1647808200  # exclusive
tz = pytz.timezone('Asia/Shanghai')

# 1) find top 30 cmdb_id + kpi_name combos by full-series row count
combo_counts = (
    df.groupby(['cmdb_id', 'kpi_name'], dropna=False)
      .size()
      .reset_index(name='total_rows_full_series')
      .sort_values('total_rows_full_series', ascending=False)
      .head(30)
)
# create a combo key for easy matching
combo_counts['combo_key'] = combo_counts['cmdb_id'].astype(str) + '||' + combo_counts['kpi_name'].astype(str)
selected_keys = set(combo_counts['combo_key'].tolist())

# prepare a column for combo_key in df to filter efficiently
df['_combo_key'] = df['cmdb_id'].astype(str) + '||' + df['kpi_name'].astype(str)

# 2) compute global P95 per selected combo using full series (no window filtering)
def compute_p95(series):
    return float(np.nanpercentile(series.dropna().values, 95)) if series.dropna().size>0 else float('nan')

p95_list = []
for key in combo_counts['combo_key']:
    cmdb, kpi = key.split('||', 1)
    mask = (df['cmdb_id'] == cmdb) & (df['kpi_name'] == kpi)
    vals = df.loc[mask, 'value']
    p95 = compute_p95(vals)
    p95_list.append(p95)
combo_counts['global_p95'] = p95_list

# 3) filter metric points to the incident window and selected combos
window_mask = (df['timestamp'].notna()) & (df['timestamp'] >= start_ts) & (df['timestamp'] < end_ts) & (df['_combo_key'].isin(selected_keys))
df_window = df.loc[window_mask, ['timestamp', 'cmdb_id', 'kpi_name', 'value', '_combo_key']].copy()

# 4) for each selected combo, find anomalies where value > global_p95
# merge p95 info into df_window
df_window = df_window.merge(combo_counts[['combo_key','global_p95','total_rows_full_series']], left_on='_combo_key', right_on='combo_key', how='left')

# mark anomalies
df_window['is_anomaly'] = df_window['value'] > df_window['global_p95']

# aggregate per combo within window
agg_funcs = {
    'timestamp': ['count', 'min'],
    'value': ['max'],
    'is_anomaly': ['sum']
}
grouped = df_window.groupby(['cmdb_id', 'kpi_name', 'combo_key', 'global_p95', 'total_rows_full_series'], dropna=False).agg(
    rows_in_window = ('timestamp', 'count'),
    earliest_timestamp_in_window = ('timestamp', 'min'),
    max_value_in_window = ('value', 'max'),
    anomaly_count_in_window = ('is_anomaly', 'sum')
).reset_index()

# We need earliest_anomaly_timestamp (min timestamp where value>global_p95). Compute separately for anomalies.
anomaly_rows = df_window[df_window['is_anomaly']].copy()
if not anomaly_rows.empty:
    earliest_anom = anomaly_rows.groupby(['cmdb_id','kpi_name','combo_key'])['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_anomaly_timestamp'})
    # merge
    grouped = grouped.merge(earliest_anom, on=['cmdb_id','kpi_name','combo_key'], how='left')
else:
    grouped['earliest_anomaly_timestamp'] = pd.NA

# keep only combos with at least one anomaly in window
grouped = grouped[grouped['anomaly_count_in_window'] > 0].copy()

# add timezone-aware ISO for earliest_anomaly_timestamp
def ts_to_iso(ts):
    if pd.isna(ts):
        return None
    return datetime.fromtimestamp(int(ts), tz).isoformat()

grouped['earliest_anomaly_timestamp_unix'] = grouped['earliest_anomaly_timestamp'].astype('Int64')
grouped['earliest_anomaly_timestamp_tz_iso'] = grouped['earliest_anomaly_timestamp_unix'].apply(lambda x: ts_to_iso(x) if pd.notna(x) else None)

# select and rename required columns
result_df = grouped[['cmdb_id', 'kpi_name', 'total_rows_full_series', 'rows_in_window',
                     'global_p95', 'earliest_anomaly_timestamp_unix', 'earliest_anomaly_timestamp_tz_iso',
                     'max_value_in_window', 'anomaly_count_in_window']].copy()

# sort by earliest_anomaly_timestamp ascending and limit to top 20
result_df = result_df.sort_values('earliest_anomaly_timestamp_unix', ascending=True).head(20)

# make numeric types plain Python floats/ints for JSON-serializable cleanliness
result_df['global_p95'] = result_df['global_p95'].astype(float)
result_df['max_value_in_window'] = result_df['max_value_in_window'].astype(float)
result_df['total_rows_full_series'] = result_df['total_rows_full_series'].astype(int)
result_df['rows_in_window'] = result_df['rows_in_window'].astype(int)
result_df['anomaly_count_in_window'] = result_df['anomaly_count_in_window'].astype(int)
result_df['earliest_anomaly_timestamp_unix'] = result_df['earliest_anomaly_timestamp_unix'].astype('Int64')

# prepare final JSON-serializable list (compact)
result_list = result_df.to_dict(orient='records')

# display compact result (as variable)
result_list

```
Out[2]:
```
```python
# Summarize the previously computed anomaly results (result_list) as a plain-English string.
# The variable 'result_list' is expected to exist in the kernel (from prior computation).
# The final displayed output is the string variable `summary`.

try:
    records = result_list
except NameError:
    records = []

def fmt_num(x):
    if x is None:
        return "null"
    try:
        x = float(x)
    except Exception:
        return str(x)
    if abs(x) >= 1000:
        return f"{x:,.0f}"
    if abs(x) >= 1:
        return f"{x:.3f}"
    return f"{x:.6f}"

lines = []
lines.append(f"Scanned top 30 cmdb_id+kpi series. Found {len(records)} series with at least one point > global P95 in the window [1647806400, 1647808200). Showing up to 20 results (sorted by earliest anomaly time):")
for r in records:
    cmdb = r.get('cmdb_id')
    kpi = r.get('kpi_name')
    total = r.get('total_rows_full_series')
    in_window = r.get('rows_in_window')
    p95 = fmt_num(r.get('global_p95'))
    earliest = r.get('earliest_anomaly_timestamp_tz_iso')
    maxv = fmt_num(r.get('max_value_in_window'))
    cnt = r.get('anomaly_count_in_window')
    lines.append(f"- {earliest} | {cmdb} | {kpi} | total_rows={total} | rows_in_window={in_window} | global_P95={p95} | max_in_window={maxv} | anomalies={cnt}")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

[{'cmdb_id': 'adservice', 'kpi_name': 'app.http.mrt', 'total_rows_full_series': 30, 'rows_in_window': 30, 'global_p95': 0.009654166666668665, 'earliest_anomaly_timestamp_unix': 1647806580, 'earliest_anomaly_timestamp_tz_iso': '2022-03-21T04:03:00+08:00', 'max_value_in_window': 0.0096770833333406, 'anomaly_count_in_window': 2}, {'cmdb_id': 'shippingservice2-0', 'kpi_name': 'container.node-5.container_network_transmit_MB.eth0', 'total_rows_full_series': 30, 'rows_in_window': 30, 'global_p95': 0.5844287395477293, 'earliest_anomaly_timestamp_unix': 1647806640, 'earliest_anomaly_timestamp_tz_iso': '2022-03-21T04:04:00+08:00', 'max_value_in_window': 0.6540055274963379, 'anomaly_count_in_window': 2}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.count', 'total_rows_full_series': 30, 'rows_in_window': 30, 'global_p95': 85.74999999999999, 'earliest_anomaly_timestamp_unix': 1647807120, 'earliest_anomaly_timestamp_tz_iso': '2022-03-21T04:12:00+08:00', 'max_value_in_window': 89.0, 'anomaly_count_in_window': 2}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.mrt', 'total_rows_full_series': 30, 'rows_in_window': 30, 'global_p95': 0.00259470609756038, 'earliest_anomaly_timestamp_unix': 1647807240, 'earliest_anomaly_timestamp_tz_iso': '2022-03-21T04:14:00+08:00', 'max_value_in_window': 0.0026250000000021, 'anomaly_count_in_window': 2}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_LastGcInfo_duration.Copy', 'total_rows_full_series': 30, 'rows_in_window': 30, 'global_p95': 1.795833333333333, 'earliest_anomaly_timestamp_unix': 1647807420, 'earliest_anomaly_timestamp_tz_iso': '2022-03-21T04:17:00+08:00', 'max_value_in_window': 1.8333333333333333, 'anomaly_count_in_window': 2}, {'cmdb_id': 'shippingservice2-0', 'kpi_name': 'container.node-5.container_network_receive_MB.eth0', 'total_rows_full_series': 30, 'rows_in_window': 30, 'global_p95': 0.1768229722976677, 'earliest_anomaly_timestamp_unix': 1647807600, 'earliest_anomaly_timestamp_tz_iso': '2022-03-21T04:20:00+08:00', 'max_value_in_window': 0.2941513061523437, 'anomaly_count_in_window': 2}, {'cmdb_id': 'shippingservice2-0', 'kpi_name': 'container.node-5.container_network_receive_packets.eth0', 'total_rows_full_series': 30, 'rows_in_window': 30, 'global_p95': 343.24999999999994, 'earliest_anomaly_timestamp_unix': 1647807600, 'earliest_anomaly_timestamp_tz_iso': '2022-03-21T04:20:00+08:00', 'max_value_in_window': 372.0, 'anomaly_count_in_window': 2}, {'cmdb_id': 'shippingservice2-0', 'kpi_name': 'container.node-5.container_network_transmit_packets.eth0', 'total_rows_full_series': 30, 'rows_in_window': 30, 'global_p95': 250.22499999999988, 'earliest_anomaly_timestamp_unix': 1647807600, 'earliest_anomaly_timestamp_tz_iso': '2022-03-21T04:20:00+08:00', 'max_value_in_window': 281.0, 'anomaly_count_in_window': 2}, {'cmdb_id': 'shippingservice2-0', 'kpi_name': 'mesh.source.shippingservice2.jaeger-collector.istio_response_bytes.http.202.', 'total_rows_full_series': 30, 'rows_in_window': 30, 'global_p95': 1144.25, 'earliest_anomaly_timestamp_unix': 1647807960, 'earliest_anomaly_timestamp_tz_iso': '2022-03-21T04:26:00+08:00', 'max_value_in_window': 1194.0, 'anomaly_count_in_window': 1}, {'cmdb_id': 'adservice', 'kpi_name': 'app.grpc.sr', 'total_rows_full_series': 30, 'rows_in_window': 30, 'global_p95': 79.91893499925628, 'earliest_anomaly_timestamp_unix': 1647808020, 'earliest_anomaly_timestamp_tz_iso': '2022-03-21T04:27:00+08:00', 'max_value_in_window': 80.68181818181817, 'anomaly_count_in_window': 2}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionCount.Copy', 'total_rows_full_series': 30, 'rows_in_window': 30, 'global_p95': 42123.8875, 'earliest_anomaly_timestamp_unix': 1647808080, 'earliest_anomaly_timestamp_tz_iso': '2022-03-21T04:28:00+08:00', 'max_value_in_window': 42136.16666666666, 'anomaly_count_in_window': 2}, {'cmdb_id': 'adservice', 'kpi_name': 'runtime.java_lang_GarbageCollector_CollectionTime.Copy', 'total_rows_full_series': 30, 'rows_in_window': 30, 'global_p95': 66162.1375, 'earliest_anomaly_timestamp_unix': 1647808080, 'earliest_anomaly_timestamp_tz_iso': '2022-03-21T04:28:00+08:00', 'max_value_in_window': 66181.0, 'anomaly_count_in_window': 2}]```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd
import pytz
from datetime import datetime

# reuse existing df (metric) and result_list from prior steps
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')

try:
    result_list
except NameError:
    # fallback: try to reconstruct from previous variable 'result_list' name variants
    result_list = []

# ensure types
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# timezone
tz = pytz.timezone('Asia/Shanghai')

# incident window
start_ts = 1647806400
end_ts = 1647808200  # exclusive

# extract combos from prior result_list
combos = []
for r in result_list:
    cmdb = r.get('cmdb_id')
    kpi = r.get('kpi_name')
    if cmdb is not None and kpi is not None:
        combos.append((cmdb, kpi))
# deduplicate while preserving order
seen = set()
combos_uniq = []
for c in combos:
    if c not in seen:
        combos_uniq.append(c)
        seen.add(c)

# helper to compute global p95 from full series
def compute_global_p95(series):
    vals = series.dropna().values
    if vals.size == 0:
        return float('nan')
    return float(np.nanpercentile(vals, 95))

# helper to convert unix ts to tz ISO
def ts_to_iso(ts):
    if pd.isna(ts):
        return None
    return datetime.fromtimestamp(int(ts), tz).isoformat()

rows = []
for cmdb, kpi in combos_uniq:
    mask_full = (df['cmdb_id'] == cmdb) & (df['kpi_name'] == kpi)
    series_full = df.loc[mask_full, 'value']
    if series_full.shape[0] == 0:
        continue
    global_p95 = compute_global_p95(series_full)
    # filter window points for this combo
    mask_window = mask_full & (df['timestamp'] >= start_ts) & (df['timestamp'] < end_ts)
    df_win = df.loc[mask_window, ['timestamp', 'value']].sort_values('timestamp').reset_index(drop=True)
    n_points_window = int(df_win.shape[0])
    if n_points_window == 0:
        continue
    # anomaly boolean per point
    df_win['is_anom'] = df_win['value'] > global_p95
    n_anom_total = int(df_win['is_anom'].sum())
    if n_anom_total == 0:
        continue
    # earliest anomaly timestamp overall in window
    earliest_anom_ts = df_win.loc[df_win['is_anom'], 'timestamp'].min()
    earliest_anom_iso = ts_to_iso(earliest_anom_ts)
    # identify consecutive anomalous segments: consider anomalies sorted by timestamp,
    # consecutive if successive anomaly timestamps differ exactly by 60 seconds.
    anom_rows = df_win.loc[df_win['is_anom'], ['timestamp', 'value']].copy().reset_index(drop=True)
    if anom_rows.shape[0] == 0:
        continue
    # compute diffs
    anom_rows['prev_ts'] = anom_rows['timestamp'].shift(1)
    anom_rows['diff'] = anom_rows['timestamp'] - anom_rows['prev_ts']
    # start of segment when diff!=60 or first row
    anom_rows['new_seg'] = (anom_rows['diff'] != 60) | (anom_rows['prev_ts'].isna())
    anom_rows['seg_id'] = anom_rows['new_seg'].cumsum()
    seg_grp = anom_rows.groupby('seg_id').agg(
        seg_start_ts = ('timestamp', 'min'),
        seg_len = ('timestamp', 'count'),
        seg_max_value = ('value', 'max')
    ).reset_index().sort_values('seg_start_ts', ascending=True)
    # pick first consecutive segment (the first row in seg_grp)
    if seg_grp.shape[0] == 0:
        continue
    first_seg = seg_grp.iloc[0]
    first_seg_len = int(first_seg['seg_len'])
    first_seg_start_ts = int(first_seg['seg_start_ts'])
    first_seg_max_value = float(first_seg['seg_max_value'])
    # only include combos whose first consecutive anomalous segment has length >= 2 minutes
    if first_seg_len < 2:
        continue
    # compute breach ratio, handle p95==0
    if global_p95 == 0 or np.isnan(global_p95):
        breach_ratio = None
    else:
        breach_ratio = float((first_seg_max_value - global_p95) / global_p95)
    rows.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_p95': float(global_p95),
        'number_of_points_in_window': n_points_window,
        'number_of_anomalous_points_in_window': n_anom_total,
        'earliest_anomaly_timestamp_tz_iso': ts_to_iso(earliest_anom_ts),
        'first_consecutive_segment_start_tz_iso': ts_to_iso(first_seg_start_ts),
        'first_consecutive_segment_length_minutes': first_seg_len,
        'first_segment_max_value': first_seg_max_value,
        'first_segment_breach_ratio': breach_ratio,
        'first_consecutive_segment_start_unix': first_seg_start_ts
    })

# build DataFrame, sort and limit top 20
result_df = pd.DataFrame(rows)
if not result_df.empty:
    result_df = result_df.sort_values('first_consecutive_segment_start_unix', ascending=True).head(20)
    # drop internal unix column before returning
    result_df = result_df[['cmdb_id','kpi_name','global_p95','number_of_points_in_window',
                           'number_of_anomalous_points_in_window','earliest_anomaly_timestamp_tz_iso',
                           'first_consecutive_segment_start_tz_iso','first_consecutive_segment_length_minutes',
                           'first_segment_max_value','first_segment_breach_ratio']].reset_index(drop=True)

# final compact result: result_df (pandas DataFrame)
result_df

```
Out[3]:
```
```python
# Summarize the computed consecutive-anomaly segments (result_df) as a plain-English string.
# Assumes result_df exists in the kernel (from previous computation).
try:
    result_df
except NameError:
    result_df = None

if result_df is None or result_df.shape[0] == 0:
    summary = "No combos found with a first consecutive anomalous segment of length >= 2 minutes in the incident window."
else:
    lines = []
    lines.append(f"Found {result_df.shape[0]} combos whose first consecutive anomalous segment lasted >= 2 minutes (window 2022-03-21 04:00:00+08 to 04:30:00+08):")
    for _, r in result_df.iterrows():
        cmdb = r['cmdb_id']
        kpi = r['kpi_name']
        p95 = float(r['global_p95'])
        pts = int(r['number_of_points_in_window'])
        anom_pts = int(r['number_of_anomalous_points_in_window'])
        earliest = r['earliest_anomaly_timestamp_tz_iso']
        seg_start = r['first_consecutive_segment_start_tz_iso']
        seg_len = int(r['first_consecutive_segment_length_minutes'])
        seg_max = float(r['first_segment_max_value'])
        breach = r['first_segment_breach_ratio']
        breach_str = ("N/A" if breach is None else f"{breach*100:.3f}%")
        lines.append(
            f"- {seg_start} | {cmdb} | {kpi} | global_P95={p95:.6g} | points_in_window={pts} | anomalous_points={anom_pts} | "
            f"earliest_anomaly={earliest} | first_segment_start={seg_start} | first_segment_length={seg_len} min | "
            f"first_segment_max={seg_max:.6g} | breach_ratio={breach_str}"
        )
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name    global_p95  number_of_points_in_window  number_of_anomalous_points_in_window earliest_anomaly_timestamp_tz_iso first_consecutive_segment_start_tz_iso  first_consecutive_segment_length_minutes  first_segment_max_value  first_segment_breach_ratio
0  shippingservice2-0  container.node-5.container_network_receive_MB....      0.176823                          30                                     2         2022-03-21T04:20:00+08:00              2022-03-21T04:20:00+08:00                                         2                 0.294151                    0.663536
1  shippingservice2-0  container.node-5.container_network_receive_pac...    343.250000                          30                                     2         2022-03-21T04:20:00+08:00              2022-03-21T04:20:00+08:00                                         2               372.000000                    0.083758
2  shippingservice2-0  container.node-5.container_network_transmit_pa...    250.225000                          30                                     2         2022-03-21T04:20:00+08:00              2022-03-21T04:20:00+08:00                                         2               281.000000                    0.122989
3           adservice  runtime.java_lang_GarbageCollector_CollectionC...  42123.887500                          30                                     2         2022-03-21T04:28:00+08:00              2022-03-21T04:28:00+08:00                                         2             42136.166667                    0.000292
4           adservice  runtime.java_lang_GarbageCollector_CollectionT...  66162.137500                          30                                     2         2022-03-21T04:28:00+08:00              2022-03-21T04:28:00+08:00                                         2             66181.000000                    0.000285```
```

In [ ]:
# In[4]:

import numpy as np
import pandas as pd
import pytz
from datetime import datetime

# reuse trace dataframe if present
try:
    df_t
except NameError:
    df_t = pd.read_csv('trace.csv')

# ensure types
df_t['timestamp'] = pd.to_numeric(df_t['timestamp'], errors='coerce').astype('Int64')
df_t['value'] = pd.to_numeric(df_t['value'], errors='coerce')

# timezone
tz = pytz.timezone('Asia/Shanghai')

# incident window
start_ts = 1647806400
end_ts = 1647808200  # exclusive

# select relevant trace rows: from_frontend OR trace.self
mask_relevant = df_t['trace_name'].str.contains('from_frontend', na=False) | df_t['trace_name'].str.contains('trace.self', na=False)
df_relevant = df_t.loc[mask_relevant].copy()

# detect metric_type and threshold type per trace_name
def detect_metric_type(name):
    n = name or ''
    if 'row_count' in n:
        return 'row_count', 'P5'
    if 'error_rate' in n:
        return 'error_rate', 'P95'
    if 'duration_p95' in n:
        return 'duration_p95', 'P95'
    if 'duration_mean' in n:
        return 'duration_mean', 'P95'
    if 'duration' in n:
        return 'duration_mean', 'P95'
    return None, None

# build unique combos (cmdb_id, trace_name) from relevant set
combos = df_relevant[['cmdb_id','trace_name']].drop_duplicates().reset_index(drop=True)

rows = []
# Precompute thresholds using full series per combo (rule: global thresholds from full series BEFORE windowing)
for _, rowc in combos.iterrows():
    cmdb = rowc['cmdb_id']
    tname = rowc['trace_name']
    metric_type, threshold_type = detect_metric_type(tname)
    if metric_type is None:
        continue
    # full series mask for this combo
    mask_full = (df_t['cmdb_id'] == cmdb) & (df_t['trace_name'] == tname)
    vals_full = df_t.loc[mask_full, 'value'].dropna()
    if vals_full.size == 0:
        continue
    if threshold_type == 'P95':
        global_thresh = float(np.nanpercentile(vals_full.values, 95))
    else:  # P5
        global_thresh = float(np.nanpercentile(vals_full.values, 5))
    # now filter window
    mask_window = mask_full & (df_t['timestamp'] >= start_ts) & (df_t['timestamp'] < end_ts)
    df_win = df_t.loc[mask_window, ['timestamp','value']].sort_values('timestamp').reset_index(drop=True)
    if df_win.shape[0] == 0:
        continue
    # determine breaches
    if threshold_type == 'P95':
        df_win['is_breach'] = df_win['value'] > global_thresh
    else:
        df_win['is_breach'] = df_win['value'] < global_thresh
    breach_count = int(df_win['is_breach'].sum())
    if breach_count == 0:
        continue
    # earliest breach timestamp
    earliest_breach_ts = int(df_win.loc[df_win['is_breach'], 'timestamp'].min())
    # compute max_or_min_value_in_window among breached points
    if threshold_type == 'P95':
        max_or_min = float(df_win.loc[df_win['is_breach'], 'value'].max())
    else:
        max_or_min = float(df_win.loc[df_win['is_breach'], 'value'].min())
    rows.append({
        'cmdb_id': cmdb,
        'trace_name': tname,
        'metric_type': metric_type,
        'threshold_type': threshold_type,
        'global_threshold': global_thresh,
        'earliest_anomaly_timestamp_unix': earliest_breach_ts,
        'earliest_anomaly_timestamp_tz_iso': datetime.fromtimestamp(earliest_breach_ts, tz).isoformat(),
        'max_or_min_value_in_window': max_or_min,
        'anomaly_count_in_window': breach_count
    })

# assemble DataFrame, sort, limit top 20, keep compact columns
breaches_df = pd.DataFrame(rows)
if not breaches_df.empty:
    breaches_df = breaches_df.sort_values('earliest_anomaly_timestamp_unix', ascending=True).head(20)
    breaches_df = breaches_df[[
        'cmdb_id','trace_name','metric_type','threshold_type','global_threshold',
        'earliest_anomaly_timestamp_tz_iso','max_or_min_value_in_window','anomaly_count_in_window'
    ]].reset_index(drop=True)

# display compact DataFrame of breaches (JSON-serializable)
breaches_df

```
Out[4]:
```
```python
# Build a concise plain-English summary of the trace breach results and display it as a string.
try:
    breaches_df
except NameError:
    # If breaches_df is not present, produce a fallback message.
    summary = "No breach results available in the current session."
else:
    n = len(breaches_df)
    earliest = breaches_df['earliest_anomaly_timestamp_tz_iso'].min()
    # Collect up to 6 illustrative examples (earliest ones)
    examples = []
    for _, r in breaches_df.head(6).iterrows():
        examples.append(f"{r['earliest_anomaly_timestamp_tz_iso']} | {r['cmdb_id']} | {r['trace_name']} | {r['metric_type']} ({r['threshold_type']}) thr={r['global_threshold']:.6g} val={r['max_or_min_value_in_window']:.6g} count={int(r['anomaly_count_in_window'])}")
    examples_text = "\n  - ".join(examples)

    summary = (
        f"Trace breaches from frontends: found {n} (cmdb_id, trace_name, metric) combos breaching their global thresholds during the incident window.\n"
        f"Earliest breach observed at {earliest}.\n\n"
        f"Summary of breach types:\n"
        f"- Many breaches are latency-related (duration_mean / duration_p95) exceeding their P95 thresholds.\n"
        f"- Several are row_count drops (below P5), indicating reduced traffic seen by services.\n"
        f"- Example earliest breaches (up to 6):\n  - {examples_text}\n\n"
        f"Notable patterns: multiple services (e.g., frontend, currencyservice, cartservice, recommendationservice, shippingservice, productcatalogservice) show either increased latency or reduced row_count starting at ~04:00–04:01+08. Recommendationservice shows particularly large duration spikes vs its P95.\n\n"
        f"Interpretation: the incident window contains widespread frontend-originated trace anomalies — both latency spikes and traffic drops — beginning at the start of the window. These combos deserve priority investigation (start with the earliest examples above)."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                           trace_name    metric_type threshold_type  global_threshold earliest_anomaly_timestamp_tz_iso  max_or_min_value_in_window  anomaly_count_in_window
0              cartservice-0   trace.from_frontend-1.duration_p95   duration_p95            P95          0.000632         2022-03-21T04:00:00+08:00                    0.001000                        2
1              cartservice-0  trace.from_frontend-1.duration_mean  duration_mean            P95          0.000124         2022-03-21T04:00:00+08:00                    0.000222                        2
2                 frontend-2                 trace.self.row_count      row_count             P5        334.800000         2022-03-21T04:00:00+08:00                  307.000000                        2
3                 frontend-1             trace.self.duration_mean  duration_mean            P95          0.004133         2022-03-21T04:00:00+08:00                    0.004182                        2
4          currencyservice-2      trace.from_frontend-2.row_count      row_count             P5         25.900000         2022-03-21T04:00:00+08:00                   22.000000                        2
5          currencyservice-1      trace.from_frontend-2.row_count      row_count             P5         25.900000         2022-03-21T04:00:00+08:00                   21.000000                        2
6          currencyservice-0      trace.from_frontend-2.row_count      row_count             P5         24.900000         2022-03-21T04:00:00+08:00                   21.000000                        2
7         checkoutservice2-0                 trace.self.row_count      row_count             P5          7.400000         2022-03-21T04:00:00+08:00                    7.000000                        2
8    recommendationservice-1             trace.self.duration_mean  duration_mean            P95          0.001917         2022-03-21T04:00:00+08:00                    0.005131                        2
9   recommendationservice2-0  trace.from_frontend2-0.duration_p95   duration_p95            P95          0.003394         2022-03-21T04:00:00+08:00                    0.003448                        2
10  recommendationservice2-0              trace.self.duration_p95   duration_p95            P95          0.003028         2022-03-21T04:00:00+08:00                    0.003083                        2
11  recommendationservice2-0             trace.self.duration_mean  duration_mean            P95          0.002347         2022-03-21T04:00:00+08:00                    0.002356                        1
12         shippingservice-0  trace.from_frontend-0.duration_mean  duration_mean            P95          0.000264         2022-03-21T04:00:00+08:00                    0.000375                        2
13   recommendationservice-0              trace.self.duration_p95   duration_p95            P95          0.002288         2022-03-21T04:00:00+08:00                    0.002307                        2
14   recommendationservice-1  trace.from_frontend-1.duration_mean  duration_mean            P95          0.002375         2022-03-21T04:00:00+08:00                    0.012727                        2
15   recommendationservice-1   trace.from_frontend-1.duration_p95   duration_p95            P95          0.002806         2022-03-21T04:00:00+08:00                    0.054410                        2
16         shippingservice-2  trace.from_frontend-0.duration_mean  duration_mean            P95          0.000194         2022-03-21T04:00:00+08:00                    0.000342                        2
17         shippingservice-0   trace.from_frontend-0.duration_p95   duration_p95            P95          0.000323         2022-03-21T04:00:00+08:00                    0.000375                        2
18   productcatalogservice-0   trace.from_frontend-2.duration_p95   duration_p95            P95          0.005656         2022-03-21T04:01:00+08:00                    0.005685                        2
19                frontend-2              trace.self.duration_p95   duration_p95            P95          0.006836         2022-03-21T04:01:00+08:00                    0.006934                        2```
```

In [ ]:
# In[5]:

import pandas as pd

# reuse existing metric dataframe 'df' if available, else load
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')

# ensure columns are present
if 'kpi_name' not in df.columns or 'cmdb_id' not in df.columns:
    raise ValueError("metric dataframe missing required columns")

# find rows where kpi_name contains 'container.node-5.' (literal substring)
mask_node5 = df['kpi_name'].fillna('').str.contains('container.node-5.', regex=False)

# group counts per cmdb_id for those kpi rows
node5_counts_df = (
    df.loc[mask_node5]
      .groupby('cmdb_id', dropna=False)
      .size()
      .reset_index(name='count')
      .sort_values('count', ascending=False)
)

# limit to top 50 entries
node5_counts_top = node5_counts_df.head(50).reset_index(drop=True)

# determine if any adservice-related cmdb_id is present
# match cmdb_id equal to 'adservice' or starting with 'adservice' (covers 'adservice-', 'adservice-0', etc.)
adservice_mask = node5_counts_top['cmdb_id'].fillna('').str.startswith('adservice')
adservice_on_node5 = bool(adservice_mask.any())

# total unique cmdbs in the node5 list
total_unique_cmdbs = int(node5_counts_df['cmdb_id'].nunique())

# assemble JSON-serializable result
node5_cmdb_counts = node5_counts_top[['cmdb_id','count']].to_dict(orient='records')

result_node5 = {
    'node5_cmdb_counts': node5_cmdb_counts,
    'adservice_on_node5': adservice_on_node5,
    'total_unique_cmdbs': total_unique_cmdbs
}

result_node5

```
Out[5]:
```
```python
# Build a concise plain-English summary string from the previous result (result_node5)
try:
    result_node5
except NameError:
    # fallback to values observed earlier
    result_node5 = {'node5_cmdb_counts': [{'cmdb_id': 'adservice-0', 'count': 1920}, {'cmdb_id': 'adservice-1', 'count': 1920}, {'cmdb_id': 'adservice-2', 'count': 1920}, {'cmdb_id': 'cartservice-0', 'count': 1920}, {'cmdb_id': 'cartservice-1', 'count': 1920}, {'cmdb_id': 'cartservice-2', 'count': 1920}, {'cmdb_id': 'checkoutservice-0', 'count': 1920}, {'cmdb_id': 'checkoutservice-1', 'count': 1920}, {'cmdb_id': 'checkoutservice-2', 'count': 1920}, {'cmdb_id': 'checkoutservice2-0', 'count': 1920}, {'cmdb_id': 'currencyservice-0', 'count': 1920}, {'cmdb_id': 'currencyservice-1', 'count': 1920}, {'cmdb_id': 'currencyservice-2', 'count': 1920}, {'cmdb_id': 'emailservice-0', 'count': 1920}, {'cmdb_id': 'emailservice-1', 'count': 1920}, {'cmdb_id': 'emailservice-2', 'count': 1920}, {'cmdb_id': 'emailservice2-0', 'count': 1920}, {'cmdb_id': 'frontend-0', 'count': 1920}, {'cmdb_id': 'frontend-1', 'count': 1920}, {'cmdb_id': 'frontend-2', 'count': 1920}, {'cmdb_id': 'paymentservice-0', 'count': 1920}, {'cmdb_id': 'paymentservice-2', 'count': 1920}, {'cmdb_id': 'productcatalogservice-0', 'count': 1920}, {'cmdb_id': 'productcatalogservice-2', 'count': 1920}, {'cmdb_id': 'productcatalogservice-1', 'count': 1920}, {'cmdb_id': 'recommendationservice-0', 'count': 1920}, {'cmdb_id': 'recommendationservice-1', 'count': 1920}, {'cmdb_id': 'shippingservice-2', 'count': 1920}, {'cmdb_id': 'recommendationservice-2', 'count': 1920}, {'cmdb_id': 'shippingservice-1', 'count': 1920}, {'cmdb_id': 'shippingservice-0', 'count': 1920}, {'cmdb_id': 'shippingservice2-0', 'count': 1920}, {'cmdb_id': 'paymentservice-1', 'count': 1800}, {'cmdb_id': 'redis-cart2-0', 'count': 240}, {'cmdb_id': 'redis-cart-0', 'count': 240}], 'adservice_on_node5': True, 'total_unique_cmdbs': 35}

top_list = result_node5['node5_cmdb_counts'][:10]
top_str = "; ".join([f"{r['cmdb_id']}({r['count']})" for r in top_list])

summary = (
    f"Found {result_node5['total_unique_cmdbs']} unique cmdb_id values with at least one kpi containing 'container.node-5.'\n"
    f"Top examples (cmdb_id(count) up to 10): {top_str}\n"
    f"adservice present on node-5? {bool(result_node5['adservice_on_node5'])}\n"
    f"Most listed services have 1,920 matching KPI rows; a few have different counts (e.g., paymentservice-1: 1,800; redis-cart*: 240)."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'node5_cmdb_counts': [{'cmdb_id': 'adservice-0', 'count': 1920}, {'cmdb_id': 'adservice-1', 'count': 1920}, {'cmdb_id': 'adservice-2', 'count': 1920}, {'cmdb_id': 'cartservice-0', 'count': 1920}, {'cmdb_id': 'cartservice-1', 'count': 1920}, {'cmdb_id': 'cartservice-2', 'count': 1920}, {'cmdb_id': 'checkoutservice-0', 'count': 1920}, {'cmdb_id': 'checkoutservice-1', 'count': 1920}, {'cmdb_id': 'checkoutservice-2', 'count': 1920}, {'cmdb_id': 'checkoutservice2-0', 'count': 1920}, {'cmdb_id': 'currencyservice-0', 'count': 1920}, {'cmdb_id': 'currencyservice-1', 'count': 1920}, {'cmdb_id': 'currencyservice-2', 'count': 1920}, {'cmdb_id': 'emailservice-0', 'count': 1920}, {'cmdb_id': 'emailservice-1', 'count': 1920}, {'cmdb_id': 'emailservice-2', 'count': 1920}, {'cmdb_id': 'emailservice2-0', 'count': 1920}, {'cmdb_id': 'frontend-0', 'count': 1920}, {'cmdb_id': 'frontend-1', 'count': 1920}, {'cmdb_id': 'frontend-2', 'count': 1920}, {'cmdb_id': 'paymentservice-0', 'count': 1920}, {'cmdb_id': 'paymentservice-2', 'count': 1920}, {'cmdb_id': 'productcatalogservice-0', 'count': 1920}, {'cmdb_id': 'productcatalogservice-2', 'count': 1920}, {'cmdb_id': 'productcatalogservice-1', 'count': 1920}, {'cmdb_id': 'recommendationservice-0', 'count': 1920}, {'cmdb_id': 'recommendationservice-1', 'count': 1920}, {'cmdb_id': 'shippingservice-2', 'count': 1920}, {'cmdb_id': 'recommendationservice-2', 'count': 1920}, {'cmdb_id': 'shippingservice-1', 'count': 1920}, {'cmdb_id': 'shippingservice-0', 'count': 1920}, {'cmdb_id': 'shippingservice2-0', 'count': 1920}, {'cmdb_id': 'paymentservice-1', 'count': 1800}, {'cmdb_id': 'redis-cart2-0', 'count': 240}, {'cmdb_id': 'redis-cart-0', 'count': 240}], 'adservice_on_node5': True, 'total_unique_cmdbs': 35}```
```

In [ ]:
# In[6]:

import numpy as np
import pandas as pd
import pytz
from datetime import datetime

# reuse existing metric dataframe 'df' if available, else load
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')

# ensure types
df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')
df['value'] = pd.to_numeric(df['value'], errors='coerce')

# timezone for ISO formatting
tz = pytz.timezone('Asia/Shanghai')
def ts_to_iso(ts):
    if pd.isna(ts):
        return None
    return datetime.fromtimestamp(int(ts), tz).isoformat()

# parameters
start_ts = 1647806400
end_ts = 1647808200  # exclusive

# 1) select series where kpi_name contains the substring
mask_net = df['kpi_name'].fillna('').str.contains('container.node-5.container_network', regex=False)
df_net = df.loc[mask_net].copy()

# if no matching rows, return empty DataFrame
if df_net.shape[0] == 0:
    result_df = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','global_p95','first_consecutive_segment_start_tz_iso',
        'first_consecutive_segment_length_minutes','first_segment_max_value',
        'first_segment_breach_ratio','anomaly_count_in_window'
    ])
else:
    # 2) compute global P95 per series using full series (no window filter) - using df_net which contains full series for these kpi_names
    p95_df = (
        df_net.groupby(['cmdb_id','kpi_name'], dropna=False)['value']
              .quantile(0.95)
              .reset_index(name='global_p95')
    )

    # 3) merge threshold back to df_net
    df_net = df_net.merge(p95_df, on=['cmdb_id','kpi_name'], how='left')

    # 4) filter to incident window
    df_window = df_net[(df_net['timestamp'] >= start_ts) & (df_net['timestamp'] < end_ts)].copy()
    df_window = df_window.sort_values(['cmdb_id','kpi_name','timestamp']).reset_index(drop=True)

    # 5) mark anomalies where value > global_p95
    df_window['is_anom'] = df_window['value'] > df_window['global_p95']

    # 6) iterate each series to find consecutive anomalous segments and collect first segment with length >=2
    rows = []
    grouped = df_window.groupby(['cmdb_id','kpi_name', 'global_p95'], sort=False)
    for (cmdb,kpi,global_p95), g in grouped:
        g = g.sort_values('timestamp').reset_index(drop=True)
        if g.shape[0] == 0:
            continue
        # total anomalies in window
        anom_total = int(g['is_anom'].sum())
        if anom_total == 0:
            continue
        # build anomalies-only rows
        anom = g.loc[g['is_anom'], ['timestamp','value']].reset_index(drop=True)
        if anom.shape[0] == 0:
            continue
        # detect consecutive segments where successive timestamps differ by exactly 60 seconds
        anom['prev_ts'] = anom['timestamp'].shift(1)
        anom['diff'] = anom['timestamp'] - anom['prev_ts']
        anom['new_seg'] = (anom['prev_ts'].isna()) | (anom['diff'] != 60)
        anom['seg_id'] = anom['new_seg'].cumsum()
        segs = anom.groupby('seg_id', sort=False).agg(
            seg_start_ts = ('timestamp','min'),
            seg_len = ('timestamp','count'),
            seg_max_value = ('value','max')
        ).reset_index().sort_values('seg_start_ts', ascending=True)
        if segs.shape[0] == 0:
            continue
        first_seg = segs.iloc[0]
        first_len = int(first_seg['seg_len'])
        first_start_ts = int(first_seg['seg_start_ts'])
        first_max_val = float(first_seg['seg_max_value'])
        # only include if first consecutive anomalous segment has length >= 2 minutes
        if first_len >= 2:
            if global_p95 == 0 or np.isnan(global_p95):
                breach_ratio = None
            else:
                breach_ratio = float((first_max_val - global_p95) / global_p95)
            rows.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'global_p95': float(global_p95),
                'first_consecutive_segment_start_unix': first_start_ts,
                'first_consecutive_segment_start_tz_iso': ts_to_iso(first_start_ts),
                'first_consecutive_segment_length_minutes': first_len,
                'first_segment_max_value': first_max_val,
                'first_segment_breach_ratio': breach_ratio,
                'anomaly_count_in_window': anom_total
            })

    # assemble DataFrame and sort/limit top 20
    result_df = pd.DataFrame(rows)
    if not result_df.empty:
        result_df = result_df.sort_values('first_consecutive_segment_start_unix', ascending=True).head(20)
        # select and order columns for compact output
        result_df = result_df[[
            'cmdb_id','kpi_name','global_p95','first_consecutive_segment_start_tz_iso',
            'first_consecutive_segment_length_minutes','first_segment_max_value',
            'first_segment_breach_ratio','anomaly_count_in_window'
        ]].reset_index(drop=True)
    else:
        result_df = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','global_p95','first_consecutive_segment_start_tz_iso',
            'first_consecutive_segment_length_minutes','first_segment_max_value',
            'first_segment_breach_ratio','anomaly_count_in_window'
        ])

# Display compact result DataFrame (JSON-serializable)
result_df

```
Out[6]:
```
```python
# Build a concise plain-English summary of the consecutive network-anomaly results (result_df).
# Assumes result_df is present in the kernel from previous computation.
try:
    result_df
except NameError:
    summary = "No results available."
else:
    n = int(result_df.shape[0])
    earliest = None
    if n > 0:
        earliest = result_df['first_consecutive_segment_start_tz_iso'].min()
    lines = []
    lines.append(f"Found {n} container.node-5.container_network.* series whose first consecutive anomalous segment lasted >= 2 minutes within the incident window.")
    if earliest:
        lines.append(f"Earliest first-segment start: {earliest}.")
    lines.append("Examples (first up to 10, showing cmdb_id | kpi_name (truncated) | P95 | start | length_min | seg_max | breach_ratio):")
    for _, r in result_df.head(10).iterrows():
        cmdb = r['cmdb_id']
        kpi = r['kpi_name']
        # truncate kpi for compactness
        kpi_short = (kpi[:80] + '...') if len(kpi) > 80 else kpi
        p95 = float(r['global_p95'])
        start = r['first_consecutive_segment_start_tz_iso']
        length = int(r['first_consecutive_segment_length_minutes'])
        seg_max = float(r['first_segment_max_value'])
        br = r['first_segment_breach_ratio']
        br_str = ("N/A" if br is None else f"{br*100:.2f}%")
        lines.append(f"- {cmdb} | {kpi_short} | P95={p95:.6g} | {start} | {length} min | max={seg_max:.6g} | breach={br_str}")
    lines.append("Interpretation: multiple services on node-5 experienced short (>=2 min) network metric spikes vs their global P95, many starting at the beginning of the incident window (~04:00+08). These network spikes are candidates for root-cause investigation.")
    summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  global_p95 first_consecutive_segment_start_tz_iso  first_consecutive_segment_length_minutes  first_segment_max_value  first_segment_breach_ratio  anomaly_count_in_window
0            emailservice-2  container.node-5.container_network_receive_pac...  325.025000              2022-03-21T04:00:00+08:00                                         2               332.000000                    0.021460                        2
1            emailservice-2  container.node-5.container_network_transmit_pa...  234.575000              2022-03-21T04:00:00+08:00                                         2               242.000000                    0.031653                        2
2            emailservice-2  container.node-5.container_network_receive_MB....    0.288308              2022-03-21T04:00:00+08:00                                         2                 0.289907                    0.005545                        2
3          paymentservice-2  container.node-5.container_network_receive_MB....    0.168801              2022-03-21T04:00:00+08:00                                         2                 0.286060                    0.694664                        2
4          paymentservice-2  container.node-5.container_network_transmit_pa...  206.675000              2022-03-21T04:00:00+08:00                                         2               215.000000                    0.040281                        2
5   productcatalogservice-2  container.node-5.container_network_receive_MB....    0.523244              2022-03-21T04:00:00+08:00                                         2                 0.589764                    0.127130                        2
6          paymentservice-2  container.node-5.container_network_receive_pac...  288.725000              2022-03-21T04:00:00+08:00                                         2               302.000000                    0.045978                        2
7           emailservice2-0  container.node-5.container_network_receive_MB....    0.169287              2022-03-21T04:03:00+08:00                                         2                 0.288995                    0.707128                        2
8           emailservice2-0  container.node-5.container_network_transmit_pa...  208.575000              2022-03-21T04:03:00+08:00                                         2               241.000000                    0.155460                        2
9   recommendationservice-2  container.node-5.container_network_receive_MB....    0.223288              2022-03-21T04:03:00+08:00                                         2                 0.338204                    0.514650                        2
10  recommendationservice-2  container.node-5.container_network_transmit_pa...  335.725000              2022-03-21T04:03:00+08:00                                         2               351.500000                    0.046988                        2
11            redis-cart2-0  container.node-5.container_network_receive_pac...  455.200000              2022-03-21T04:05:00+08:00                                         2               468.000000                    0.028120                        2
12            redis-cart2-0  container.node-5.container_network_receive_MB....    0.180642              2022-03-21T04:05:00+08:00                                         2                 0.300356                    0.662715                        2
13            redis-cart2-0  container.node-5.container_network_transmit_pa...  282.575000              2022-03-21T04:05:00+08:00                                         2               302.000000                    0.068743                        2
14  recommendationservice-1  container.node-5.container_network_receive_MB....    0.224719              2022-03-21T04:06:00+08:00                                         2                 0.345671                    0.538240                        2
15             redis-cart-0  container.node-5.container_network_receive_MB....    0.194697              2022-03-21T04:06:00+08:00                                         2                 0.313380                    0.609575                        2
16             redis-cart-0  container.node-5.container_network_transmit_pa...  343.875000              2022-03-21T04:06:00+08:00                                         2               378.000000                    0.099237                        2
17             redis-cart-0  container.node-5.container_network_receive_pac...  570.475000              2022-03-21T04:06:00+08:00                                         2               603.500000                    0.057890                        2
18           emailservice-0  container.node-5.container_network_receive_pac...  290.875000              2022-03-21T04:07:00+08:00                                         2               329.500000                    0.132789                        2
19           emailservice-0  container.node-5.container_network_receive_MB....    0.168555              2022-03-21T04:07:00+08:00                                         2                 0.289469                    0.717360                        2```
```